# Pairs Trading with Mean-Reverting Stochastic Processes
**MAC F244 project (Topic 17)**: Ornstein-Uhlenbeck spreads, ADF/Engle-Granger pair selection and optimal trading bands, on US and Indian stocks (2010-2026).

**How to use this notebook (temporary master copy)**
- Every section is tagged with its owner (M1-M5). Edit only your own sections, then tick your task in `Project_Tracker.csv`.
- The maths and the backtest live in `src/`; this notebook calls them and makes every figure and table in the report.
- Figures are saved to `figs/` as PNG and EPS with `savefig(fig, name)`, using the names `report/main.tex` expects.
- Cells marked **HEAVY** run the full walk-forward backtests (slow the first time, cached in `results/cache/` afterwards). They only run when `RUN_HEAVY = True`.
- Before submission, M1 runs *Restart & Run All* once and commits the outputs.

| Section | Content | Owner |
| --- | --- | --- |
| 1 | Setup | M1 |
| 2 | Data | M2 |
| 3 | OU model and simulation | M3 |
| 4 | E1 estimator accuracy, E2a ADF power, E2b test size | M3, M4, M2 |
| 5 | Pair selection on one window | M2 |
| 6 | Trading rules; Bertram bands and E3 | M4; M5 |
| 7 | Backtest engine checks, E4 synthetic pair | M1 |
| 8 | E5 headline backtest | M1, M4 |
| 9 | E8 India | M2, M5 |
| 10 | E9 cointegration persistence | M2 |
| 11 | Conclusions | M3 |

## 1. Setup  (Owner: M1)
Imports, the fixed seed (42) and the switch for the heavy backtests. All parameters live in `src/config.py`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.dates
import matplotlib.pyplot as plt

from src import config as C
from src.data import load_prices, benchmark_returns, sectors
from src.ou import (simulate_ou, fit_ou, bertram_cycle_time, bertram_bands,
                    bertram_return_rate, first_passage_cycle_mc)
from src.cointegration import adf_test, engle_granger, engle_granger_both, naive_residual_adf
from src.selection import select_cointegrated, select_distance
from src.backtest import run_selection, simulate, pair_returns
from src.metrics import compute_metrics, metrics_table
from src.plots import savefig, COLORS

np.random.seed(C.SEED)
rng = np.random.default_rng(C.SEED)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

RUN_HEAVY = True    # True runs the full walk-forward backtests (sections 8-13)
STRATS = ['distance', 'zscore', 'sscore', 'bertram']

## 2. Data  (Owner: M2)
Daily adjusted closes from Yahoo Finance (`yfinance`), cached in `data/raw/` (see `data/raw/README.md` for retrieval date and flagged jumps). 56 US stocks in 7 sectors plus EWA/EWC, and 26 NSE stocks in 5 sectors. Pairs are formed only within a sector.

In [2]:
px_us = load_prices('us')
px_in = load_prices('india')
bench_us = benchmark_returns('us')
bench_in = benchmark_returns('india')

summary = pd.DataFrame({
    m: {'tickers': p.shape[1], 'first day': p.index[0].date(), 'last day': p.index[-1].date(),
        'trading days': len(p), 'missing %': round(100 * p.isna().mean().mean(), 2)}
    for m, p in {'US': px_us, 'India': px_in}.items()}).T
summary

,tickers,first day,last day,trading days,missing %
US,58,2010-01-04,2026-09-29,4210,0.000
India,26,2012-01-02,2026-09-29,3637,0.000


In [3]:
# T08 (M2): cleaning and jump report
from src.data import jump_report, RAW, UNIVERSES, MAX_GAP_DAYS
for m, px in [('us', px_us), ('india', px_in)]:
    raw = pd.read_csv(RAW / f'{m}_prices.csv', index_col=0, parse_dates=True).sort_index()
    assert px.index.is_monotonic_increasing and UNIVERSES[m]['benchmark'] not in px.columns
    filled = int((raw[px.columns].isna() & px.notna()).sum().sum())     # cells repaired by the forward fill
    print(f'{m.upper()}: {filled} gaps of up to {MAX_GAP_DAYS} days forward-filled; '
          f'{100 * px.isna().mean().mean():.2f}% of cells missing after cleaning')
jumps = pd.concat({'US': jump_report(px_us), 'India': jump_report(px_in)}, names=['market']).reset_index(level=0)
jumps['return'] = jumps['return'].map('{:+.1%}'.format)
jumps   # one-day moves above 25%, listed for inspection

US: 0 gaps of up to 3 days forward-filled; 0.00% of cells missing after cleaning
INDIA: 0 gaps of up to 3 days forward-filled; 0.00% of cells missing after cleaning


,market,date,ticker,return
0,US,2020-03-09,EOG,-32.0%
1,US,2020-03-09,HAL,-37.6%
2,US,2020-03-09,OXY,-52.0%
3,US,2020-03-09,SLB,-27.4%
4,US,2020-03-24,COP,+25.2%
5,US,2020-03-24,HAL,+26.5%
6,US,2020-06-05,OXY,+33.7%
7,US,2020-11-09,VLO,+31.2%
8,US,2024-08-02,INTC,-26.1%
9,US,2024-08-29,DG,-32.1%


**Data checks (M2, T08).** `load_prices` sorts by date, forward-fills gaps of at most 3 trading days and drops the benchmark. In this download no gap needed filling and no cell is missing. Moves above 25% in one day are listed rather than removed: 12 in the US and 4 in India. Those checked match real events (for example the 9 March 2020 oil-price crash; see `data/raw/README.md`), not split or spin-off errors, so no price was edited. `tests/test_data.py` tests the loader, the cleaning rule and the jump report.

In [4]:
# Two classic pairs, prices normalised to 1 at the start
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (a, b) in zip(axes, [('KO', 'PEP'), ('XOM', 'CVX')]):
    p = px_us[[a, b]].dropna()
    (p / p.iloc[0]).plot(ax=ax, lw=1)
    ax.set_title(f'{a} vs {b}'); ax.set_ylabel('Normalised price'); ax.set_xlabel('')
savefig(fig, 'fig_data_pairs')

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


## 3. The Ornstein-Uhlenbeck model  (Owner: M3)
The log-price spread $X_t = \ln P^A_t - \beta \ln P^B_t - \mu$ is modelled as
$$dX_t = \kappa(\theta - X_t)\,dt + \sigma\,dW_t, \qquad \kappa > 0.$$
Itô's lemma on $e^{\kappa t}X_t$ gives
$$X_t = \theta + (X_0-\theta)e^{-\kappa t} + \sigma\int_0^t e^{-\kappa(t-s)}\,dW_s,$$
so $X_t$ is Gaussian with mean $\theta + (X_0-\theta)e^{-\kappa t}$, variance $\sigma^2(1-e^{-2\kappa t})/(2\kappa)$, stationary law $N(\theta, \sigma^2/2\kappa)$ and half-life $\ln 2/\kappa$.

**Exact discretisation.** With step $\Delta = 1/252$:
$$X_{t+\Delta} = a + bX_t + \varepsilon_t,\quad b = e^{-\kappa\Delta},\quad a = \theta(1-b),\quad \mathrm{Var}(\varepsilon_t) = \sigma^2\frac{1-e^{-2\kappa\Delta}}{2\kappa}.$$
OLS of $X_{t+\Delta}$ on $X_t$ is the conditional MLE, and $\hat\kappa = -\ln\hat b/\Delta$, $\hat\theta = \hat a/(1-\hat b)$, $\sigma_{eq} = \hat s_\varepsilon/\sqrt{1-\hat b^2}$.

**Euler-Maruyama** uses $X_{t+\Delta} = X_t + \kappa(\theta - X_t)\Delta + \sigma\sqrt{\Delta}Z$, i.e. $b \approx 1-\kappa\Delta$. The noise is additive, so Milstein's correction term is zero and Milstein equals Euler-Maruyama.

**Checked against report section 2 (M3, T05).** Every equation above matches the report: the Itô solution, its mean and variance, the stationary law, the exact AR(1) step and the estimators. The check cell below confirms the estimator recovers known parameters on long simulated paths.

In [5]:
# Exact vs Euler-Maruyama paths driven by the same shocks (same seed)
kappa, theta, sigma, n = 25.0, 0.0, 0.2, 252
x_exact = simulate_ou(kappa, theta, sigma, 0.08, n, method='exact', rng=np.random.default_rng(1))[0]
x_euler = simulate_ou(kappa, theta, sigma, 0.08, n, method='euler', rng=np.random.default_rng(1))[0]
sig_eq = sigma / np.sqrt(2 * kappa)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(x_exact, lw=1, label='Exact scheme')
ax.plot(x_euler, lw=1, ls='--', label='Euler-Maruyama')
ax.axhspan(theta - sig_eq, theta + sig_eq, color='grey', alpha=0.15, label=r'$\theta \pm \sigma_{eq}$')
ax.set_xlabel('Trading day'); ax.set_ylabel('Spread'); ax.legend()
savefig(fig, 'fig_ou_paths')

# T11: the estimator recovers known parameters on a long exact path
for k in [5, 25, 50]:
    x = simulate_ou(k, 0.0, 0.2, 'stationary', 252 * 40, rng=np.random.default_rng(C.SEED + k))[0]
    p = fit_ou(x)
    print(f'kappa={k:>3}: kappa_hat={p.kappa:6.2f}, theta_hat={p.theta:+.4f}, sigma_hat={p.sigma:.4f}, half-life={p.half_life:5.1f} d')

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


kappa=  5: kappa_hat=  5.65, theta_hat=-0.0051, sigma_hat=0.2014, half-life= 30.9 d
kappa= 25: kappa_hat= 24.91, theta_hat=-0.0002, sigma_hat=0.1981, half-life=  7.0 d
kappa= 50: kappa_hat= 49.31, theta_hat=+0.0003, sigma_hat=0.2015, half-life=  3.5 d


## 4. E1 and E2: can we estimate and detect mean reversion?  (Owners: M3 E1, M4 E2a, M2 E2b)
**E1.** Simulate an OU spread with a known speed $\kappa$, estimate $\kappa$ from 63 to 504 days of data, and repeat 1,000 times to see how far off the estimate is on average.

**E2a.** The ADF regression $\Delta X_t = \alpha + \gamma X_{t-1} + \sum_i\delta_i\Delta X_{t-i} + \varepsilon_t$ has $\gamma = e^{-\kappa\Delta}-1$, so testing $\gamma=0$ is testing $\kappa=0$ (no mean reversion). We measure how often ADF detects a mean-reverting spread for different half-lives and window lengths, which tells us what a 12-month formation window can find.

**E2b.** The test we use on a pair is run on regression residuals, whose coefficient was fitted to make them look stationary, so plain ADF critical values are too lenient (spurious regression). We check this on two independent random walks, which are never cointegrated: a correct 5% test should flag about 5% of them.

In [6]:
# E1: bias and RMSE of kappa_hat
kappas, lengths, n_paths = [5, 10, 25, 50], [63, 126, 252, 504], 1000
rng_e1 = np.random.default_rng(C.SEED)   # own seed: same numbers whatever ran before
rows = []
for k in kappas:
    for T in lengths:
        x = simulate_ou(k, 0.0, 0.2, 'stationary', T, n_paths=n_paths, rng=rng_e1)
        k_exact = np.array([fit_ou(p).kappa for p in x])
        k_euler = np.array([fit_ou(p, estimator='euler').kappa for p in x])
        rows.append(dict(kappa=k, days=T, valid_share=np.isfinite(k_exact).mean(),
                         rel_bias_exact=np.nanmean(k_exact) / k - 1,
                         rel_rmse_exact=np.sqrt(np.nanmean((k_exact - k) ** 2)) / k,
                         rel_bias_euler=np.nanmean(k_euler) / k - 1))
e1 = pd.DataFrame(rows)
e1.to_csv('results/e1_ou_estimation.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 3.6))
for k in kappas:
    d = e1[e1.kappa == k]
    ax.plot(d.days, 100 * d.rel_bias_exact, marker='o', label=f'kappa = {k} / yr')
ax.set_xscale('log'); ax.set_xticks(lengths); ax.set_xticklabels(lengths); ax.minorticks_off()
ax.set_xlabel('Window length (trading days)'); ax.set_ylabel('Relative bias of kappa_hat (%)'); ax.legend()
savefig(fig, 'fig_e1_ou_estimation')
e1

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


,kappa,days,valid_share,rel_bias_exact,rel_rmse_exact,rel_bias_euler
0,5,63,0.984,4.331,5.793,3.932
1,5,126,0.987,1.962,2.805,1.840
2,5,252,0.998,0.960,1.430,0.912
3,5,504,1.000,0.458,0.767,0.433
4,10,63,0.994,2.146,3.050,1.878
5,10,126,1.000,0.939,1.437,0.845
6,10,252,1.000,0.460,0.799,0.411
7,10,504,1.000,0.206,0.439,0.174
8,25,63,1.000,0.742,1.225,0.562
9,25,126,1.000,0.372,0.694,0.268


In [7]:
# E2a: ADF power over half-life x window length (rejection rate at 5%)
half_lives, lengths, reps = [5, 10, 20, 40, 80], [63, 126, 252, 504], 300
rng_e2 = np.random.default_rng(C.SEED)   # own seed: same numbers whatever ran before
power = pd.DataFrame(index=half_lives, columns=lengths, dtype=float)
for hl in half_lives:
    k = np.log(2) / (hl * C.DT)
    for T in lengths:
        x = simulate_ou(k, 0.0, 0.2, 'stationary', T, n_paths=reps, rng=rng_e2)
        power.loc[hl, T] = np.mean([adf_test(p, maxlag=5)[1] < 0.05 for p in x])
power.to_csv('results/e2_adf_power.csv')

fig, ax = plt.subplots(figsize=(5.5, 3.6))
im = ax.imshow(power.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(lengths)), lengths); ax.set_yticks(range(len(half_lives)), half_lives)
ax.set_xlabel('Window length (trading days)'); ax.set_ylabel('True half-life (days)')
for i in range(len(half_lives)):
    for j in range(len(lengths)):
        ax.text(j, i, f'{power.values[i, j]:.2f}', ha='center', va='center', color='k' if power.values[i, j] > 0.6 else 'w', fontsize=8)
fig.colorbar(im, label='ADF rejection rate')
ax.grid(False)
savefig(fig, 'fig_e2_adf_power')

# Report version, drawn at its printed size (0.40 of the text width) so the labels print at 6-7 pt
with plt.rc_context({'font.size': 7, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'legend.fontsize': 6}):
    fig, ax = plt.subplots(figsize=(2.75, 1.95))
    im = ax.imshow(power.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(lengths)), lengths); ax.set_yticks(range(len(half_lives)), half_lives)
    ax.set_xlabel('Window length (trading days)'); ax.set_ylabel('True half-life (days)')
    for i in range(len(half_lives)):
        for j in range(len(lengths)):
            ax.text(j, i, f'{power.values[i, j]:.2f}', ha='center', va='center', fontsize=6,
                    color='k' if power.values[i, j] > 0.6 else 'w')
    fig.colorbar(im, label='ADF rejection rate'); ax.grid(False)
    savefig(fig, 'fig_e2_adf_power_report')
power

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

... (26 outputs left out)

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

,63,126,252,504
5,0.297,0.717,0.990,1.000
10,0.163,0.253,0.737,0.997
20,0.133,0.127,0.303,0.730
40,0.087,0.080,0.133,0.257
80,0.097,0.057,0.090,0.100


In [8]:
# E2b (M2, T30): size of the residual test when the truth is 'not cointegrated'
# Two independent random walks of 252 days (the formation window); both tests have a nominal 5% level.
rng_e2b = np.random.default_rng(C.SEED)   # own seed: same numbers whatever ran before
reps, n = 1000, 252
naive, eg = [], []
for _ in range(reps):
    y, x = np.cumsum(rng_e2b.standard_normal((2, n)) * 0.01, axis=1)
    naive.append(naive_residual_adf(y, x)[1] < 0.05)
    eg.append(engle_granger(y, x).pvalue < 0.05)
e2_size = pd.DataFrame({'Rejection rate (nominal 5%)': [np.mean(naive), np.mean(eg)]},
                       index=['Plain ADF critical values on residuals', 'Engle-Granger (MacKinnon) critical values'])
e2_size['Monte Carlo SE'] = np.sqrt(e2_size.iloc[:, 0] * (1 - e2_size.iloc[:, 0]) / reps)
e2_size.to_csv('results/e2_size.csv')
e2_size

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

... (16 outputs left out)

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

,Rejection rate (nominal 5%),Monte Carlo SE
Plain ADF critical values on residuals,0.169,0.012
Engle-Granger (MacKinnon) critical values,0.062,0.008


**E1 (M3).** Estimating $\kappa$ is harder than detecting it. With one year of data $\hat\kappa$ is too high by about 18% for a pair with a 7-day half-life ($\kappa = 25$) and by about 96% for a slow pair ($\kappa = 5$), and the error roughly halves each time the window doubles. So fitted half-lives look shorter than they really are. (results/e1_ou_estimation.csv)

**E2a (M4).** With one year of data, ADF almost always detects a spread with a 5-day half-life (99% of the time) and usually one with a 10-day half-life (74%), but rarely a 20-day one (30%), and slower ones hardly more often than chance. So the formation window can only find fast pairs, and indeed every pair we select on real data has a half-life of 21 days or less.

**E2b (M2).** (results/e2_size.csv) On 1,000 pairs of independent random walks of 252 days, where the true answer is "not cointegrated", the plain ADF test on OLS residuals rejects in 16.9% of pairs at a nominal 5%, more than three times too often. Engle-Granger with MacKinnon critical values rejects in 6.2% (Monte Carlo SE 0.8 points), close to nominal (slightly above, within about 1.6 SE).

- **Why the naive test over-rejects.** OLS picks $\hat\beta$ to make the residuals as small as possible, so even for unrelated random walks the residuals look more stationary than a randomly drawn series (the spurious regression of Granger & Newbold 1974). The Dickey-Fuller distribution assumes the series was not chosen this way; MacKinnon's critical values are shifted to account for the fitted $\hat\beta$.
- **Consequence for pair selection.** With plain ADF about one random pair in six would pass in a single direction, so a screen of 196 sector pairs would flag up to about 33 spurious pairs per window against about 10 for Engle-Granger (fewer in both cases, since both directions must pass). This is why `engle_granger` calls `coint`, and why `naive_residual_adf` is kept only for this experiment (`tests/test_cointegration.py` checks the same gap).
- **Caveat.** A correct 5% test still flags about 10 of 196 independent pairs by chance, and the pass counts we observe (about 8 per window in the US, E5) are no higher than that. This is why E5 finds so few genuine pairs, and why Benjamini-Hochberg is reported.

## 5. Pair selection on one formation window  (Owner: M2)
Engle-Granger in both directions (keep a pair only if both p-values are below 5%), positive hedge ratio, half-life 2-63 trading days, at least 12 mean crossings a year, ranked by the Engle-Granger statistic, top 20. On the latest window (Sep 2025 - Sep 2026) only 3 of 196 candidate pairs pass, fewer than the 10 a 5% test would flag by chance.

In [9]:
# T13 (M2): Engle-Granger screen of the 196 US sector pairs on the latest formation window
form = np.log(px_us.iloc[-C.FORMATION_DAYS:]).dropna(axis=1)
print(f'Formation window: {form.index[0].date()} to {form.index[-1].date()}')
chosen, diag, tested = select_cointegrated(form, sectors('us'))
print(diag)
n_t = diag['n_tested']
print(f"{diag['n_eg']} of {n_t} pairs pass Engle-Granger at 5% in both directions (chance alone: about {0.05 * n_t:.1f}); "
      f"{diag['n_bh']} survive Benjamini-Hochberg (q = {C.BH_Q}, smallest p must be below {C.BH_Q / n_t:.4f}); "
      f"{diag['n_filtered']} pass every filter")
chosen_ids = {(s.a, s.b) for s in chosen}
top = (pd.DataFrame([dict(pair=f'{s.a}/{s.b}', sector=s.sector, beta=s.beta, eg_stat=s.eg_stat,
                          eg_p=s.eg_pvalue, half_life_days=s.half_life, crossings_per_year=s.crossings,
                          selected=(s.a, s.b) in chosen_ids) for s in tested])
       .sort_values('eg_p').head(10).reset_index(drop=True))
top.index += 1
top.to_csv('results/t13_top10_pairs.csv', index_label='rank')
top   # the ten pairs with the smallest Engle-Granger p-value (larger of the two directions)

Formation window: 2025-09-29 to 2026-09-29


{'n_tested': 196, 'n_eg': 3, 'n_bh': 0, 'n_filtered': 3}
3 of 196 pairs pass Engle-Granger at 5% in both directions (chance alone: about 9.8); 0 survive Benjamini-Hochberg (q = 0.1, smallest p must be below 0.0005); 3 pass every filter


,pair,sector,beta,eg_stat,eg_p,half_life_days,crossings_per_year,selected
1,JPM/BAC,Banks,0.802,-3.917,0.010,4.083,52.000,True
2,TXN/INTC,Technology,0.470,-3.465,0.036,7.298,33.000,True
3,PEP/HSY,Consumer staples,0.565,-3.826,0.043,8.715,22.000,True
4,USB/PNC,Banks,0.992,-3.403,0.050,7.127,41.000,False
5,NEE/DUK,Utilities,1.173,-3.604,0.051,8.588,36.000,False
6,OXY/EOG,Energy,1.123,-3.291,0.057,9.739,27.000,False
7,XEL/SO,Utilities,0.748,-3.485,0.069,6.858,43.000,False
8,PH/ITW,Industrials,1.213,-3.458,0.074,10.268,17.000,False
9,GS/PNC,Banks,0.962,-3.389,0.078,7.729,30.000,False
10,NEE/XEL,Utilities,1.035,-3.506,0.080,11.779,20.000,False


**T13 result (M2) for the proposal's progress paragraph.** (results/t13_top10_pairs.csv) On the latest formation window (29 Sep 2025 to 29 Sep 2026), **3 of the 196 within-sector US pairs pass Engle-Granger at 5%** in both directions, against about 10 expected by chance alone if no pair were cointegrated. All three also pass the OU filters and are the pairs traded next: JPM/BAC (p = 0.010, half-life 4 days), TXN/INTC (p = 0.036, 7 days) and PEP/HSY (p = 0.043, 9 days). None survives Benjamini-Hochberg at q = 0.10, because the smallest p-value (0.010) is far above the 0.0005 that the rule requires for the best of 196 tests. The next seven pairs in the list miss 5% with p-values from 0.050 (USB/PNC) to 0.083 (GS/PNC). The honest summary is that this window offers little evidence of cointegration beyond what chance produces, which matches the average over all windows (about 8 of 196 pairs pass in E5, no more than chance) and E9 shows that selected pairs mostly do not stay cointegrated out of sample. The two classic pairs of the proposal figure, KO/PEP and EWA/EWC, are not among them (T12).

**Check of the distance baseline S0 (M2, T29).** Gatev et al. (2006) normalise each price to 1 at the start of the formation window, rank within-sector pairs by the sum of squared differences of the normalised prices, open when the difference exceeds 2 formation standard deviations and close when the normalised prices cross. We check `select_distance` and the S0 rule against that definition: (1) a toy case worked by hand; (2) whether log prices (our code) rank pairs like price indices (Gatev); (3) where the spread's zero line sits. Our code centres the spread on the last formation-day gap, while "prices cross" means the first-day gap. (3) is run as a sensitivity only; the headline S0 row of Table 1 keeps the code as it is.

In [10]:
# T29 (M2): S0 distance baseline against Gatev et al. (2006)
from dataclasses import replace
from src.selection import candidate_pairs

# (1) Toy case worked by hand: three log-price series, one sector
toy = pd.DataFrame({'a': [0.0, 0.1, 0.2, 0.1], 'b': [0.0, 0.1, 0.1, 0.1], 'c': [0.0, 0.5, 1.0, 0.0]})
top = select_distance(toy, {'S': ['a', 'b', 'c']}, k=2)
assert [(s.a, s.b) for s in top] == [('a', 'b'), ('a', 'c')]      # SSD 0.01 and 0.81 (b/c is 0.98)
assert abs(top[0].ssd - 0.01) < 1e-12 and abs(top[1].ssd - 0.81) < 1e-12
assert top[0].beta == 1.0 and abs(top[0].mu) < 1e-12
print('Toy case: smallest SSD pair is a/b with SSD', top[0].ssd)

if RUN_HEAVY:
    sel_us = run_selection('us', px_us, sectors('us'))        # cached after the first run
    logp_all = np.log(px_us)

    # (2) Gatev's price index vs our log prices: do they pick the same 20 pairs?
    overlap = []
    for sel in sel_us:
        w = sel['window']
        form = logp_all.iloc[w.f0:w.t0]
        form = form.loc[:, form.isna().mean() <= 0.02].ffill().bfill()
        idx = np.exp(form - form.iloc[0])
        ssd = {(i, j): float(((idx[i] - idx[j]) ** 2).sum()) for _, i, j in candidate_pairs(sectors('us'), set(form.columns))}
        top_idx = set(sorted(ssd, key=ssd.get)[:C.TOP_K])
        overlap.append(len(top_idx & {(s.a, s.b) for s in sel['distance']}) / C.TOP_K)
    print(f'Top-{C.TOP_K} overlap, price index vs log prices: mean {np.mean(overlap):.2f}, min {np.min(overlap):.2f} over {len(overlap)} windows')

    # (3) Where is the spread's zero line? Same pairs, three centrings
    def recentre(selections, how):
        out = []
        for sel in selections:
            w = sel['window']
            new = []
            for d in sel['distance']:
                gap = logp_all[d.a].iloc[w.f0:w.t0] - logp_all[d.b].iloc[w.f0:w.t0]
                mu = {'first': gap.iloc[0], 'mean': gap.mean(), 'last': gap.iloc[-1]}[how]
                new.append(replace(d, mu=float(mu)))
            out.append(dict(sel, distance=new))
        return out

    rows = {}
    for label, how in [('Code: zero at last formation gap (headline)', 'last'),
                       ('Gatev: zero where prices cross (first gap)', 'first'),
                       ('Zero at formation mean gap', 'mean')]:
        r = simulate('distance', px_us, recentre(sel_us, how), C.COST_BPS['us'])
        m = compute_metrics(r.committed)
        rows[label] = {'Ann. return %': m['Ann. return %'], 'Sharpe': m['Sharpe'], 'Sharpe SE (Lo)': m['Sharpe SE (Lo)'],
                       'Trades': len(r.trades), 'Hit rate %': 100 * (r.trades['pnl'] > 0).mean()}
    s0_check = pd.DataFrame(rows).T
    s0_check.to_csv('results/s0_check.csv')
    z_end = [abs((logp_all[d.a].iloc[sel['window'].t0 - 1] - logp_all[d.b].iloc[sel['window'].t0 - 1]
                  - (logp_all[d.a].iloc[sel['window'].f0] - logp_all[d.b].iloc[sel['window'].f0])) / d.sd)
             for sel in sel_us for d in sel['distance']]
    print(f'With the first-gap centring, {100 * np.mean(np.array(z_end) > 2):.0f}% of pairs are already beyond 2 sigma when trading starts')
    display(s0_check.round(2))

Toy case: smallest SSD pair is a/b with SSD 0.010000000000000002


Top-20 overlap, price index vs log prices: mean 0.89, min 0.75 over 32 windows


With the first-gap centring, 26% of pairs are already beyond 2 sigma when trading starts


,Ann. return %,Sharpe,Sharpe SE (Lo),Trades,Hit rate %
Code: zero at last formation gap (headline),-0.090,-0.050,0.250,651.000,53.460
Gatev: zero where prices cross (first gap),-0.340,-0.180,0.250,713.000,49.370
Zero at formation mean gap,-0.350,-0.180,0.250,703.000,50.360


**S0 check result (M2).** (results/s0_check.csv)

- **Pair ranking matches Gatev.** Ranking by log prices instead of price indices keeps on average 89% of the top 20 pairs (never below 75%), so the choice of normalisation is not what drives S0.
- **The zero line differs.** Our code centres the spread on the last formation-day gap, so every pair starts flat and trades only after a fresh 2-sigma move. Gatev's "prices cross" centring starts from the first-day gap; 26% of pairs are then already beyond 2 sigma on day one of trading. S0 is negative either way but worse under Gatev's centring (Sharpe -0.18 against -0.05, 713 trades against 651); both are far inside one Lo standard error (0.25) of zero, so the ranking of the four rules is unchanged.
- **Decision.** Table 1 keeps the code's centring (a pair that has just formed is treated as in equilibrium, which is the more conservative reading) and the report states the difference. The alternative is in results/s0_check.csv.

## 6. Trading rules and Bertram's optimal bands  (Owners: M4 rules, M5 Bertram and E3)
- **S0 distance** (Gatev et al. 2006): open when normalised prices diverge by 2 formation standard deviations, close at the crossing.
- **S1 z-score**: open at $|z|>2$, close at $|z|<0.5$.
- **S2 s-score** (Avellaneda & Lee 2010): $s = (X-\hat\theta)/\sigma_{eq}$; open long $s<-1.25$, open short $s>1.25$, close long $s>-0.50$, close short $s<0.75$.
- **S3 Bertram (2010)**: for the centred OU with entry $a$, exit $m$ and round-trip cost $c$,
$$\mathbb{E}[T] = \frac{\pi}{\kappa}\left[\mathrm{erfi}\left(\frac{m\sqrt\kappa}{\sigma}\right) - \mathrm{erfi}\left(\frac{a\sqrt\kappa}{\sigma}\right)\right],\qquad \mu(a,m) = \frac{m-a-c}{\mathbb{E}[T]},$$
from the backward equation $\tfrac12\sigma^2u'' - \kappa x u' = -1$. With $a=-m$ we maximise $\mu$ numerically.

**Notes on Avellaneda & Lee (2010) and Bertram (2010) (M4, T06)**
1. *Avellaneda & Lee* trade the residual of each US stock's returns after regressing on sector ETFs or PCA factors; the cumulated residual is fitted as an OU process on a 60-day window.
2. Their trading signal is the s-score $s=(X-m)/\sigma_{eq}$; thresholds 1.25 to open and 0.50 / 0.75 to close were chosen once, not optimised per stock, and only stocks with fast mean reversion ($\kappa > 252/30$, reversion time under half the window) are traded.
3. They report Sharpe ratios above 1 for 1997-2007 that fall after 2002 and around the August 2007 quant crisis, i.e. the edge decays as it becomes crowded. We use their thresholds unchanged on pairs (S2).
4. *Bertram* solves for the expected round-trip time of an OU process between an entry band $a$ and exit band $m$ via first-passage times (the erfi formula above), and the variance of the return per trade.
5. Maximising expected return per unit time gives symmetric bands $a=-m$ whose width grows with the transaction cost; maximising the Sharpe ratio instead gives wider bands. We implement the return-maximising version (S3).

**Notes on Bertram (2010) (M5, T49; full viva note in notes/M5_Bertram_viva_note.md)**
1. Bertram models the centred spread as an OU process and treats a trade as one cycle: enter at $a$, exit at $m$, wait to return to $a$.
2. The expected cycle time comes from first-passage times, which solve the backward equation $\tfrac12\sigma^2u'' - \kappa x u' = -1$; integrating twice gives the erfi formula above.
3. Dividing the profit per cycle, $m - a - c$, by $\mathbb{E}[T]$ gives the return per unit time; with symmetric bands its maximiser $m^*$ depends only on $c/\sigma_{eq}$, not on $\kappa$, and shrinks to 0 when trading is free.

In [11]:
# T43 (M4): check the z-score (S1) and s-score (S2) rules on a toy spread worked out by hand
from src.strategies import positions
from src.selection import PairSpec

toy = np.array([0, 1.0, 2.1, 1.5, 0.6, 0.4, -0.3, -2.2, -1.0, -0.6, -0.4, 0.2, 2.5, -2.5, 0.0])
unit = PairSpec('A', 'B', 'toy', 'coint', beta=1.0, mu=0.0, sd=1.0, theta=0.0, sigma_eq=1.0)
s1, s2 = positions('zscore', toy, unit), positions('sscore', toy, unit)
assert (s1 == [0, 0, -1, -1, -1, 0, 0, 1, 1, 1, 0, 0, -1, 1, 0]).all()   # open |z|>2, close inside 0.5
assert (s2 == [0, 0, -1, -1, 0, 0, 0, 1, 1, 1, 0, 0, -1, 1, 0]).all()   # 1.25 / -0.50 / 0.75
assert s2[4] == 0 and s1[4] == -1        # at 0.6 the S2 short closes (< 0.75) but S1 stays open (> 0.5)
# S2 centres on the OU mean theta and scales by sigma_eq; S1 does not
shifted = PairSpec('A', 'B', 'toy', 'coint', 1.0, 0.0, 1.0, theta=0.5, sigma_eq=0.5)
assert (positions('sscore', toy, shifted) == positions('sscore', (toy - 0.5) / 0.5, unit)).all()
# Stop-loss at |s| = 4 closes the trade and blocks re-entry until s is back inside the entry band
assert (positions('sscore', np.array([0, 2.0, 4.5, 3.0, 1.0, 2.0]), unit, stop=4.0) == [0, -1, 0, 0, 0, -1]).all()

# On simulated OU spreads (half-life 10 days) both rules earn money before costs out of sample
pnl = {'zscore': [], 'sscore': []}
for x in simulate_ou(np.log(2) / (10 * C.DT), 0, 0.2, 'stationary', 504, n_paths=200, rng=np.random.default_rng(C.SEED)):
    ou = fit_ou(x[:252])                                      # fit on the first year, trade the second
    spec = PairSpec('A', 'B', 'sim', 'coint', 1.0, 0.0, x[:252].std(), theta=ou.theta, sigma_eq=ou.sigma_eq)
    for r in pnl:
        p = positions(r, x[252:], spec)
        pnl[r].append((p[:-1] * np.diff(x[252:])).sum())
pd.DataFrame({r: {'mean spread P&L': np.mean(v), 'share of paths > 0': np.mean(np.array(v) > 0)} for r, v in pnl.items()}).T

,mean spread P&L,share of paths > 0
zscore,0.175,0.905
sscore,0.291,1.000


In [12]:
# Proposal figure (T12): spread as an s-score with the S2 bands, latest formation year.
# Shading shows where the s-score rule would hold the spread (red short, green long).
from src.strategies import positions
from src.selection import PairSpec
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, (a, b) in zip(axes, [('KO', 'PEP'), ('EWA', 'EWC')]):
    lp = np.log(px_us[[a, b]].iloc[-C.FORMATION_DAYS:].dropna())
    best, p_both = engle_granger_both(lp[a].values, lp[b].values, a, b)
    ou = fit_ou(best.resid)
    s = (best.resid - ou.theta) / ou.sigma_eq
    spec = PairSpec(best.y, best.x, '', 'coint', best.beta, best.mu, best.resid.std(),
                    theta=ou.theta, sigma_eq=ou.sigma_eq)
    pos = positions('sscore', best.resid, spec)
    ax.fill_between(lp.index, -3, 3, where=pos > 0, color='tab:green', alpha=0.12, lw=0)
    ax.fill_between(lp.index, -3, 3, where=pos < 0, color='tab:red', alpha=0.12, lw=0)
    ax.plot(lp.index, s, lw=1, color='k')
    ax.axhline(1.25, color='tab:red', lw=0.8, label='open short / long (±1.25)')
    ax.axhline(-1.25, color='tab:green', lw=0.8)
    ax.axhline(0.75, color='tab:red', ls=':', lw=0.8, label='close short (0.75)')
    ax.axhline(-0.50, color='tab:green', ls=':', lw=0.8, label='close long (-0.50)')
    sign = '-' if best.beta >= 0 else '+'
    ax.set_title(f'{best.y} {sign} {abs(best.beta):.2f} {best.x}: EG p = {p_both:.2f}, '
                 f'half-life {ou.half_life:.0f} d', fontsize=10)
    ax.set_ylim(-3, 3); ax.tick_params(axis='x', labelrotation=30)
axes[0].set_ylabel('s-score')
axes[0].legend(fontsize=7, loc='lower left')
savefig(fig, 'fig_proposal_spread')

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [13]:
# T50 (M5): check bertram_cycle_time() and bertram_bands() against the theory
from scipy.integrate import quad
kappa, sigma = 20.0, 0.1
sig_eq = sigma / np.sqrt(2 * kappa)

# (1) The erfi formula is the solution of the backward equation (1/2) sigma^2 u'' - kappa x u' = -1.
#     Up-crossing a -> m plus down-crossing m -> a reduces to (2 / sigma^2) * int_a^m s(y) dy * int s(z)^-1 dz,
#     with scale density s(y) = exp(kappa y^2 / sigma^2). Integrate that numerically and compare.
gauss = quad(lambda z: np.exp(-kappa * z * z / sigma ** 2), -np.inf, np.inf)[0]
rows = []
for m in [0.25, 0.5, 1.0, 2.0]:
    mm = m * sig_eq
    T_quad = 2 / sigma ** 2 * quad(lambda y: np.exp(kappa * y * y / sigma ** 2), -mm, mm)[0] * gauss
    T_erfi = bertram_cycle_time(-mm, mm, kappa, sigma)
    rows.append({'m / sigma_eq': m, 'quadrature (days)': 252 * T_quad, 'erfi formula (days)': 252 * T_erfi})
    assert abs(T_quad / T_erfi - 1) < 1e-8

# (2) The optimiser finds the maximum of mu(-m, m) = (2m - c) / E[T]: compare with a fine grid
for c in [0.2, 0.5, 1.0]:
    cc = c * sig_eq
    grid = np.linspace(cc / 2 * 1.0001, 4 * sig_eq, 20000)
    best = grid[np.argmax(bertram_return_rate(grid, kappa, sigma, cc))]
    assert abs(bertram_bands(kappa, sigma, cc) - best) < 1e-3 * sig_eq

# (3) Shape: m* rises with the cost and tends to 0 when trading is free (Bertram 2010)
m_star = [bertram_bands(kappa, sigma, c * sig_eq) / sig_eq for c in [0, 0.1, 0.5, 1.0, 2.0]]
assert m_star[0] < 0.01 and np.all(np.diff(m_star) > 0)
print('m* / sigma_eq at cost 0, 0.1, 0.5, 1, 2 sigma_eq:', np.round(m_star, 3))
pd.DataFrame(rows)

m* / sigma_eq at cost 0, 0.1, 0.5, 1, 2 sigma_eq: [0.    0.542 0.964 1.264 1.714]


,m / sigma_eq,quadrature (days),erfi formula (days)
0,0.250,15.958,15.958
1,0.500,32.950,32.950
2,1.000,75.482,75.482
3,2.000,298.711,298.711


In [14]:
# E3 (M5): Bertram's E[T] against simulated cycles, and m* against cost
# Each simulation covers the same 300 years of OU time, so a finer monitoring step does not mean fewer cycles.
# The cycle -m -> m -> -m is detected only at monitoring times; coarse monitoring misses crossings.
kappa, sigma = 20.0, 0.1
sig_eq = sigma / np.sqrt(2 * kappa)
rng_e3 = np.random.default_rng(C.SEED)       # own seed: same numbers whatever ran before
years = 300
m_grid = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0])
steps_per_day = {'1 day': 1, '1/10 day': 10, '1/200 day': 200}
rows = []
for m in m_grid:
    row = {'m / sigma_eq': m, 'formula (days)': 252 * bertram_cycle_time(-m * sig_eq, m * sig_eq, kappa, sigma)}
    for name, k in steps_per_day.items():
        cyc = 252 * first_passage_cycle_mc(kappa, sigma, m * sig_eq, n_steps=years * 252 * k, dt=C.DT / k, rng=rng_e3)
        row[f'MC {name}'] = cyc.mean()
        row[f'SE {name}'] = cyc.std(ddof=1) / np.sqrt(len(cyc))
    rows.append(row)
e3 = pd.DataFrame(rows)
e3.to_csv('results/e3_bertram_check.csv', index=False)

costs = np.linspace(0, 1.0, 21) * sig_eq
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(e3['m / sigma_eq'], e3['formula (days)'], color='k', label='Bertram formula')
for name, mk in [('1/200 day', 'o'), ('1 day', 's')]:
    axes[0].errorbar(e3['m / sigma_eq'], e3[f'MC {name}'], yerr=2 * e3[f'SE {name}'], fmt=mk, ms=4,
                     capsize=2, label=f'Monte Carlo, checked every {name}')
axes[0].set_xlabel('Band half-width m / sigma_eq'); axes[0].set_ylabel('Expected cycle (days)'); axes[0].legend(fontsize=8)
# m* / sigma_eq depends only on cost / sigma_eq: kappa scales E[T] but not the argmax. Speed sets the return instead.
m_star = np.array([bertram_bands(kappa, sigma, c) for c in costs])
axes[1].plot(costs / sig_eq, m_star / sig_eq, color='k', label='Optimal m* (any kappa)')
axes[1].set_xlabel('Round-trip cost / sigma_eq'); axes[1].set_ylabel('Optimal m* / sigma_eq')
ax_r = axes[1].twinx()
for k, ls in [(5, ':'), (20, '--'), (50, '-.')]:
    sigma_k = sig_eq * np.sqrt(2 * k)   # same sigma_eq, different speed
    assert np.allclose([bertram_bands(k, sigma_k, c) for c in costs], m_star, rtol=1e-4, atol=1e-6 * sig_eq)
    ax_r.plot(costs / sig_eq, bertram_return_rate(m_star, k, sigma_k, costs) / sig_eq, ls=ls, color='tab:blue',
              label=f'return per year at m*, kappa = {k}')
ax_r.set_ylabel('Return per year / sigma_eq', color='tab:blue'); ax_r.grid(False)
h1, l1 = axes[1].get_legend_handles_labels(); h2, l2 = ax_r.get_legend_handles_labels()
axes[1].legend(h1 + h2, l1 + l2, fontsize=7, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
savefig(fig, 'fig_e3_bertram')
e3.round(2)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


,m / sigma_eq,formula (days),MC 1 day,SE 1 day,MC 1/10 day,SE 1/10 day,MC 1/200 day,SE 1/200 day
0,0.250,15.960,31.760,0.520,20.470,0.310,16.730,0.250
1,0.500,32.950,51.560,0.920,37.290,0.610,34.420,0.570
2,0.750,52.220,76.430,1.530,56.870,1.010,51.730,0.880
3,1.000,75.480,101.820,2.080,83.830,1.800,78.250,1.530
4,1.500,146.340,198.670,6.780,160.910,4.370,148.660,3.750
5,2.000,298.710,457.220,19.150,327.130,14.630,304.750,12.110


**E3 result (M5).** The formula is correct: it matches a direct numerical integration (check above), and simulated cycles get closer to it the more often the spread is checked. Checked once a day, cycles take 35-99% longer than the formula says, because the spread can cross a band and come back between two checks, so the formula is optimistic for a strategy that trades on daily closes. The best band depends only on the cost relative to $\sigma_{eq}$, not on $\kappa$ (right panel).

## 7. Backtest engine checks and E4 synthetic pair  (Owner: M1)
**Timing rules (no look-ahead).** Pairs, tickers and every parameter come from the 252-day formation window only. The signal computed at the close of day $t$ is executed at the close of $t+1$ (the one-day wait of Gatev et al. 2006), so day $t$'s return is earned by the position decided at the close of $t-2$. Open positions are closed on the last day of the 126-day trading window.

**P&L.** One unit of spread is \$1 long $A$ and \$$\beta$ short $B$. Since $r^A-\beta r^B \approx d\ln P^A-\beta\,d\ln P^B = dX$, the position earns the change of the OU spread. Daily return on gross capital: $\text{pos}_{t-1}(r^A_t-\beta r^B_t)/(1+|\beta|)$, minus the cost per leg on every unit traded (a flip long$\to$short is two units).

**Checks.** `tests/test_backtest.py` and `tests/test_metrics.py` hold 20 tests with hand-computable answers. The decisive one scrambles every price after a date (and deletes some) and checks that no earlier return changes.

In [15]:
# T14, T19-T21: unit tests of the engine and the metrics, then the look-ahead check by hand
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], capture_output=True, text=True).stdout.strip().splitlines()[-1])

desired = np.zeros(30); desired[10:20] = 1          # signal on at the close of day 10
ra = np.full(30, 0.01); rb = np.zeros(30)
net, held = pair_returns(desired, 1.0, ra, rb, cost=0.0)
assert held[10] == 0 and held[11] == 1, 'execution lag broken'
assert net[11] == 0 and net[12] > 0, 'return earned before the position was held'
print('Look-ahead check passed: signal at close 10, bought at close 11, first return on day 12')

60 passed in 7.53s
Look-ahead check passed: signal at close 10, bought at close 11, first return on day 12


**Integration check (T22).** One real pair through the whole pipeline on one window: selection on 2018, trading in the first half of 2019.

In [16]:
from src.backtest import make_windows
w = make_windows(len(px_us))[16]
chosen, diag, _ = select_cointegrated(np.log(px_us.iloc[w.f0:w.t0]), sectors('us'))
spec = chosen[0]
print(f'Formation {px_us.index[w.f0].date()} to {px_us.index[w.t0 - 1].date()}, trading {px_us.index[w.t0].date()} to {px_us.index[w.t1 - 1].date()}')
print(f'{diag["n_eg"]}/{diag["n_tested"]} pairs pass Engle-Granger, {diag["n_filtered"]} pass all filters')
print(f'Top pair {spec.a}/{spec.b} ({spec.sector}): beta = {spec.beta:.3f}, EG p = {spec.eg_pvalue:.4f}, '
      f'kappa = {spec.kappa:.1f}/yr, half-life = {spec.half_life:.1f} d, sigma_eq = {spec.sigma_eq:.4f}')
one = [dict(window=w, coint=[spec], distance=[], diag=diag)]
res_one = simulate('sscore', px_us, one, C.COST_BPS['us'], k=1)

lp = np.log(px_us[[spec.a, spec.b]].iloc[w.f0:w.t1])
s = (lp[spec.a] - spec.beta * lp[spec.b] - spec.mu - spec.theta) / spec.sigma_eq
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(s.index, s.values, lw=0.9, color=COLORS['sscore'])
ax.axvspan(px_us.index[w.t0], px_us.index[w.t1 - 1], color='grey', alpha=0.12)
ax.text(px_us.index[w.t0 + 3], -3.0, 'trading window: parameters frozen', fontsize=8)
ax.set_ylim(-3.3, 2.7)
for lvl in (1.25, -1.25): ax.axhline(lvl, color='grey', lw=0.8)
for lvl in (0.75, -0.5): ax.axhline(lvl, color='grey', lw=0.8, ls=':')
for _, t in res_one.trades.iterrows():
    ax.axvspan(t.open, t.close, ymin=0.92, ymax=1.0, color='k' if t.side > 0 else 'grey')
ax.set_ylabel('s-score'); ax.set_title(f'{spec.a} - {spec.beta:.2f} {spec.b}: bars on top mark trades (black long, grey short)')
savefig(fig, 'fig_m1_integration')
print(compute_metrics(res_one.committed, trades=res_one.trades)['Total return %'], '% on the pair over the window')
res_one.trades

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Formation 2018-01-05 to 2019-01-07, trading 2019-01-08 to 2019-07-09
6/196 pairs pass Engle-Granger, 5 pass all filters
Top pair XEL/AEP (Utilities): beta = 1.023, EG p = 0.0024, kappa = 38.1/yr, half-life = 4.6 d, sigma_eq = 0.0079
1.263954332574757 % on the pair over the window


,pair,open,close,side,days,pnl
0,w16:XEL/AEP,2019-01-28,2019-02-06,1,7,0.002
1,w16:XEL/AEP,2019-03-08,2019-03-25,-1,11,0.003
2,w16:XEL/AEP,2019-04-17,2019-05-07,1,13,0.005
3,w16:XEL/AEP,2019-06-03,2019-06-19,1,12,0.005
4,w16:XEL/AEP,2019-07-01,2019-07-05,-1,3,-0.003


**E4 (T23): does the pipeline earn what the OU theory predicts?** A synthetic pair with a known OU spread ($\kappa = 25$/yr, half-life 7 days, $\sigma_{eq} = 0.03$, $\beta = 1.2$, 5 bp per leg) is pushed through the pipeline 200 times. For this pair Bertram's formula predicts the return of the optimal bands exactly, so we add one real-world friction at a time and measure what each costs:

| Level | What changes |
| --- | --- |
| L0 | Bertram's formula: continuous monitoring, true parameters |
| L1 | spread seen once a day, traded at the same close, true parameters |
| L2 | + one-day execution lag |
| L3 | + 6-month trading windows (start flat, forced close) |
| L4a | + OU parameters $\kappa,\theta,\sigma$ estimated on the formation year (true $\beta$) |
| L4 | + $\beta$ and $\mu$ estimated too |
| L5 | + pair traded only when it passes Engle-Granger and the OU filters (the full pipeline) |

In [17]:
# E4 ladder: 200 synthetic pairs, about 90 s; cached in results/
import json, os
from src.synthetic import run_ladder
if os.path.exists('results/e4_ladder.csv') and os.path.exists('results/e4_theory.json'):
    e4 = pd.read_csv('results/e4_ladder.csv'); e4_info = json.load(open('results/e4_theory.json'))
else:
    e4, e4_info = run_ladder(reps=200)
    e4.to_csv('results/e4_ladder.csv', index=False)
    json.dump({k: float(v) for k, v in e4_info.items()}, open('results/e4_theory.json', 'w'), indent=1)

levels = ['L1', 'L2', 'L3', 'L4a', 'L4', 'L5']
names = {'L0': 'L0 Bertram formula', 'L1': 'L1 daily monitoring', 'L2': 'L2 + one-day lag', 'L3': 'L3 + 6-month windows',
         'L4a': 'L4a + estimated OU params', 'L4': 'L4 + estimated beta', 'L5': 'L5 + EG selection (pipeline)'}
n = len(e4)
table_e4 = pd.DataFrame({'Ann. return %': [100 * e4_info['theory']] + [100 * e4[l].mean() for l in levels],
                         'SE %': [np.nan] + [100 * e4[l].std() / np.sqrt(n) for l in levels]},
                        index=[names[l] for l in ['L0'] + levels])
steps = [e4['L1'] - e4_info['theory']] + [e4[b] - e4[a] for a, b in zip(levels, levels[1:])]
table_e4['Step %'] = [np.nan] + [100 * d.mean() for d in steps]
table_e4['Step SE %'] = [np.nan, np.nan] + [100 * d.std() / np.sqrt(n) for d in steps[1:]]   # paired SE
table_e4['Share of theory %'] = 100 * table_e4['Ann. return %'] / table_e4['Ann. return %'].iloc[0]
table_e4.to_csv('results/e4_table.csv')
print(f"Fine-grid simulation of L0 (50 steps a day, 200 years): {100 * e4_info['fine_grid']:.1f}% vs formula {100 * e4_info['theory']:.1f}%")
print(f"Pair selected in {100 * e4.selected_share.mean():.0f}% of windows; median kappa_hat / kappa = {e4.kappa_hat_over_true.mean():.2f}; "
      f"median |beta_hat - beta| = {e4.beta_abs_err.mean():.3f}; m*/sigma_eq true {e4_info['m_star']:.3f}, fitted {e4.m_hat_in_sig_eq.mean():.3f}")
print(f"s-score on the same pipeline (L5): {100 * e4.sscore_L5.mean():.1f}%")

fig, ax = plt.subplots(figsize=(7, 3.4))
y = np.arange(len(table_e4))[::-1]
ax.barh(y, table_e4['Ann. return %'], height=0.6, color=COLORS['bertram'], xerr=2 * table_e4['SE %'].fillna(0),
        error_kw=dict(lw=1, capsize=2, ecolor='k'))
ax.axvline(100 * e4_info['theory'], color='k', lw=0.8, ls='--')
for yi, v, se in zip(y, table_e4['Ann. return %'], table_e4['SE %'].fillna(0)):
    ax.text(v + 2 * se + 0.4, yi, f'{v:.1f}%', va='center', fontsize=8, bbox=dict(fc='white', ec='none', pad=0.5))
ax.set_yticks(y, table_e4.index); ax.set_xlabel('Annual return on gross capital (%), mean of 200 pairs, ±2 SE')
ax.set_xlim(0, 100 * e4_info['theory'] * 1.15); ax.grid(axis='y', visible=False)
savefig(fig, 'fig_e4_synthetic')

# Report version at its printed size (0.57 of the text width)
short = ['L0 theory', 'L1 daily checks', 'L2 + one-day lag', 'L3 + 6-month windows', 'L4a + estimated OU',
         'L4 + estimated beta', 'L5 + selection']
with plt.rc_context({'font.size': 7, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'legend.fontsize': 6}):
    fig, ax = plt.subplots(figsize=(3.95, 1.95))
    ax.barh(y, table_e4['Ann. return %'], height=0.6, color=COLORS['bertram'], xerr=2 * table_e4['SE %'].fillna(0),
            error_kw=dict(lw=0.8, capsize=1.5, ecolor='k'))
    ax.axvline(100 * e4_info['theory'], color='k', lw=0.7, ls='--')
    for yi, v, se in zip(y, table_e4['Ann. return %'], table_e4['SE %'].fillna(0)):
        ax.text(v + 2 * se + 0.4, yi, f'{v:.1f}%', va='center', fontsize=6, bbox=dict(fc='white', ec='none', pad=0.3))
    ax.set_yticks(y, short); ax.set_xlabel('Annual return (%), mean of 200 pairs, ±2 SE')
    ax.set_xlim(0, 100 * e4_info['theory'] * 1.18); ax.grid(axis='y', visible=False)
    savefig(fig, 'fig_e4_synthetic_report')
table_e4.round(2)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Fine-grid simulation of L0 (50 steps a day, 200 years): 24.3% vs formula 24.2%
Pair selected in 74% of windows; median kappa_hat / kappa = 1.24; median |beta_hat - beta| = 0.064; m*/sigma_eq true 0.487, fitted 0.503
s-score on the same pipeline (L5): 9.7%


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


,Ann. return %,SE %,Step %,Step SE %,Share of theory %
L0 Bertram formula,24.160,NaN,NaN,NaN,100.000
L1 daily monitoring,23.080,0.130,-1.080,NaN,95.520
L2 + one-day lag,20.910,0.130,-2.170,0.090,86.530
L3 + 6-month windows,20.440,0.130,-0.470,0.030,84.600
L4a + estimated OU params,19.470,0.140,-0.980,0.100,80.560
L4 + estimated beta,17.200,0.160,-2.260,0.140,71.200
L5 + EG selection (pipeline),12.750,0.210,-4.460,0.140,52.760


**E4 result.** The fine-grid simulation reproduces Bertram's formula (24.3% vs 24.2% a year), so the formula and our P&L bookkeeping agree. The full pipeline earns 12.7% (SE 0.2%), about half the theoretical return, and every step of the gap has a cause:

* **Daily monitoring (−1.1 pp).** The spread can cross a band and come back within a day, so some cycles are missed. Each daily step moves $0.42\,\sigma_{eq}$, almost as much as the band half-width $0.49\,\sigma_{eq}$.
* **One-day lag (−2.2 pp).** By the time we trade, the spread has already reverted by about $1-e^{-\kappa\Delta}\approx 9\%$ of its distance from the mean. This is the price of an honest execution rule.
* **Window resets (−0.5 pp).** Each window starts flat and ends with a forced close.
* **Estimated OU parameters (−1.0 pp).** $\hat\kappa$ is biased upward by about 24% at 252 days (Yu 2012, our E1), but the optimal band in units of $\sigma_{eq}$ barely moves ($0.50$ vs $0.49$), so the loss is small.
* **Estimated hedge ratio (−2.3 pp).** A median error of only $|\hat\beta-\beta| = 0.064$ adds $(\beta-\hat\beta)\ln P^B$ to the traded spread: a random walk of typical size $0.064\times0.015\times\sqrt{126}\approx 0.36\,\sigma_{eq}$ over a trading window, drifting against bands only $0.49\,\sigma_{eq}$ wide. Hedge-ratio error costs more than all OU-parameter errors together (motivates a time-varying hedge ratio, such as a Kalman filter).
* **Selection (−4.5 pp).** Engle-Granger at 5% in both directions recognises this truly cointegrated pair in only 74% of formation windows (the power limit measured in E2), and the capital sits idle otherwise: $0.74 \times 17.2\% = 12.7\%$, exactly L5.

The s-score rule on the same pipeline earns 9.7%, below Bertram's 12.7%: on a true OU pair the bands built for maximum return per unit time do win. A single 12-year run is noisy (standard deviation 2.9% across pairs; the first seed we tried gave 8.6%), which is why E4 averages 200 pairs.

## 8. E5 headline backtest, US  (Owners: M1 engine, M4 strategies)  HEAVY
Rolling 12-month formation and 6-month trading windows, K = 20 pairs, 10 bp per leg. Design choices are frozen on 2010-2021; 2022-2026 is the untouched test block.

In [18]:
if RUN_HEAVY:
    sel_us = run_selection('us', px_us, sectors('us'))     # slow once, then cached
    res_us = {s: simulate(s, px_us, sel_us, C.COST_BPS['us'], label=s) for s in STRATS}
    table1 = metrics_table(res_us.values(), bench_us).rename(index=C.STRATEGY_LABELS)
    table1.to_csv('results/table1_us.csv')
    split = pd.DataFrame({
        (C.STRATEGY_LABELS[s], blk): compute_metrics(r.committed[sl])
        for s, r in res_us.items()
        for blk, sl in [('2011-2021', slice(None, '2021-12-31')), ('2022-2026 test', slice(C.TEST_START, None))]
    }).T[['Ann. return %', 'Sharpe', 'Sharpe SE (Lo)', 'Max DD %']]
    split.to_csv('results/table1_split.csv')
    display(table1.round(2)); display(split.round(2))

,Ann. return %,Ann. vol %,Sharpe,Sharpe SE (Lo),Sharpe SE (HAC),NW t-stat,Sortino,Max DD %,Calmar,Total return %,Years,Market beta,Market corr,Trades,Hit rate %,Avg days held
S0 Distance (Gatev 2006),-0.090,1.820,-0.050,0.250,0.230,-0.210,-0.070,-5.780,-0.020,-1.620,15.710,0.020,0.160,651.000,53.460,54.990
S1 Cointegration z-score,0.330,1.690,0.190,0.250,0.220,0.860,0.280,-3.940,0.080,5.000,15.710,0.020,0.230,424.000,62.740,47.080
S2 OU s-score (Avellaneda-Lee 2010),0.100,1.800,0.060,0.250,0.220,0.250,0.080,-4.160,0.020,1.320,15.710,0.030,0.240,750.000,68.930,31.590
S3 OU optimal bands (Bertram 2010),0.120,1.930,0.060,0.250,0.220,0.270,0.090,-4.840,0.020,1.560,15.710,0.020,0.220,618.000,63.920,50.810


Ann. return %  Sharpe  \
S0 Distance (Gatev 2006)            2011-2021               0.000   0.000   
                                    2022-2026 test         -0.300  -0.160   
S1 Cointegration z-score            2011-2021               0.040   0.020   
                                    2022-2026 test          0.990   0.690   
S2 OU s-score (Avellaneda-Lee 2010) 2011-2021              -0.020  -0.010   
                                    2022-2026 test          0.370   0.240   
S3 OU optimal bands (Bertram 2010)  2011-2021               0.090   0.040   
                                    2022-2026 test          0.180   0.100   

                                                    Sharpe SE (Lo)  Max DD %  
S0 Distance (Gatev 2006)            2011-2021                0.300    -5.780  
                                    2022-2026 test           0.460    -3.340  
S1 Cointegration z-score            2011-2021                0.300    -3.940  
                                    2022-2026 test           0.460    -2.330  
S2 OU s-score (Avellaneda-Lee 2010) 2011-2021                0.300    -4.160  
                                    2022-2026 test           0.460    -2.940  
S3 OU optimal bands (Bertram 2010)  2011-2021                0.300    -4.310  
                                    2022-2026 test           0.460    -3.790

In [19]:
if RUN_HEAVY:
    fig, ax = plt.subplots(figsize=(9, 3.8))
    for s, r in res_us.items():
        ax.plot((1 + r.committed).cumprod(), lw=1.2, color=COLORS[s], label=C.STRATEGY_LABELS[s])
    ax.axvspan(pd.Timestamp(C.TEST_START), px_us.index[-1], color='grey', alpha=0.12, label='Test block')
    ax.set_ylabel('Wealth on committed capital (start = 1)'); ax.legend(fontsize=8)
    savefig(fig, 'fig_e5_cumulative')

    # Report version at its printed size (0.37 of the text width), beside Table 1
    short_lab = {'distance': 'S0 distance', 'zscore': 'S1 z-score', 'sscore': 'S2 s-score', 'bertram': 'S3 Bertram'}
    with plt.rc_context({'font.size': 7, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'legend.fontsize': 6}):
        fig, ax = plt.subplots(figsize=(2.55, 1.8))
        for s, r in res_us.items():
            ax.plot((1 + r.committed).cumprod(), lw=0.8, color=COLORS[s], label=short_lab[s])
        ax.axvspan(pd.Timestamp(C.TEST_START), px_us.index[-1], color='grey', alpha=0.12)
        lo, hi = ax.get_ylim(); ax.set_ylim(lo, hi + 0.45 * (hi - lo))      # headroom for the legend
        ax.set_ylabel('Growth of $1'); ax.legend(fontsize=5.5, loc='upper left', ncol=2, handlelength=1.2, columnspacing=0.8)
        ax.xaxis.set_major_locator(mpl.dates.YearLocator(5)); ax.xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y'))
        savefig(fig, 'fig_e5_cumulative_report')

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [20]:
if RUN_HEAVY:
    # M4: why the committed-capital returns are small. Few pairs pass, so most of the K = 20 slots are idle,
    # and costs take most of the gross edge.
    diag_us = pd.DataFrame([s['diag'] for s in sel_us])
    print(f"Per window: {diag_us.n_tested.mean():.0f} pairs tested, {diag_us.n_eg.mean():.1f} pass Engle-Granger "
          f"(chance alone at 5%: up to {0.05 * diag_us.n_tested.mean():.1f}), {diag_us.n_bh.mean():.1f} survive Benjamini-Hochberg, "
          f"{diag_us.n_filtered.mean():.1f} pass all filters (of K = {C.TOP_K} slots)")
    e5_diag = pd.DataFrame({
        C.STRATEGY_LABELS[s]: {
            'Sharpe, 10 bp (committed)': compute_metrics(r.committed)['Sharpe'],
            'Sharpe, 0 bp (committed)': compute_metrics(simulate(s, px_us, sel_us, 0).committed)['Sharpe'],
            'Sharpe, 10 bp (employed)': compute_metrics(r.employed)['Sharpe'],
            'Trades per pair-window': len(r.trades) / max(len(r.pairs), 1),
        } for s, r in res_us.items()}).T
    e5_diag.to_csv('results/e5_diagnostics.csv'); display(e5_diag.round(2))

Per window: 196 pairs tested, 8.4 pass Engle-Granger (chance alone at 5%: up to 9.8), 0.6 survive Benjamini-Hochberg, 8.1 pass all filters (of K = 20 slots)


,"Sharpe, 10 bp (committed)","Sharpe, 0 bp (committed)","Sharpe, 10 bp (employed)",Trades per pair-window
S0 Distance (Gatev 2006),-0.050,0.180,-0.080,1.020
S1 Cointegration z-score,0.190,0.350,0.010,1.640
S2 OU s-score (Avellaneda-Lee 2010),0.060,0.320,-0.220,2.900
S3 OU optimal bands (Bertram 2010),0.060,0.230,-0.140,2.390


**Is any difference real? (M1)** A Sharpe ratio estimated from 16 years of data has a standard error of about $1/\sqrt{16} = 0.25$, so differences between rules can easily be noise. We test each OU rule against the distance baseline with a circular block bootstrap of the Sharpe difference (21-day blocks, the same days for both rules; Ledoit & Wolf 2008), and correct the three p-values with Holm's method because we run three tests.

In [21]:
if RUN_HEAVY:
    from src.metrics import sharpe_diff_bootstrap
    rows = {}
    for s in ['zscore', 'sscore', 'bertram']:
        for blk, sl in [('2011-2026', slice(None)), ('2022-2026 test', slice(C.TEST_START, None))]:
            rows[(C.STRATEGY_LABELS[s], blk)] = sharpe_diff_bootstrap(res_us[s].committed[sl], res_us['distance'].committed[sl])
    sig = pd.DataFrame(rows).T
    for blk in sig.index.get_level_values(1).unique():          # Holm step-down over the three rules
        p = sig.xs(blk, level=1)['p_value'].sort_values()
        adj = np.minimum(1, np.maximum.accumulate(p.values * np.arange(len(p), 0, -1)))
        for name, a in zip(p.index, adj):
            sig.loc[(name, blk), 'p_Holm'] = a
    sig.to_csv('results/significance_vs_distance.csv'); display(sig.round(3))

sharpe_1  sharpe_2  diff  \
S1 Cointegration z-score            2011-2026          0.192    -0.048 0.240   
                                    2022-2026 test     0.687    -0.163 0.850   
S2 OU s-score (Avellaneda-Lee 2010) 2011-2026          0.055    -0.048 0.104   
                                    2022-2026 test     0.240    -0.163 0.403   
S3 OU optimal bands (Bertram 2010)  2011-2026          0.061    -0.048 0.109   
                                    2022-2026 test     0.105    -0.163 0.268   

                                                    ci_low  ci_high  p_value  \
S1 Cointegration z-score            2011-2026       -0.255    0.762    0.344   
                                    2022-2026 test  -0.213    1.934    0.115   
S2 OU s-score (Avellaneda-Lee 2010) 2011-2026       -0.389    0.617    0.679   
                                    2022-2026 test  -0.650    1.484    0.458   
S3 OU optimal bands (Bertram 2010)  2011-2026       -0.390    0.611    0.661   
                                    2022-2026 test  -0.859    1.413    0.643   

                                                    p_Holm  
S1 Cointegration z-score            2011-2026        1.000  
                                    2022-2026 test   0.344  
S2 OU s-score (Avellaneda-Lee 2010) 2011-2026        1.000  
                                    2022-2026 test   0.917  
S3 OU optimal bands (Bertram 2010)  2011-2026        1.000  
                                    2022-2026 test   0.917

**How big is look-ahead bias here? (M1)** The same selected pairs, traded three ways: honestly; with execution at the signal's own close (lag 0); and with $\beta$, $\mu$ and the OU parameters re-estimated on the trading window itself, i.e. on prices not yet seen (`peeking_selections`, demonstration only).

In [22]:
if RUN_HEAVY:
    from src.backtest import peeking_selections
    peek = peeking_selections(sel_us, px_us)
    rows = {}
    for s in STRATS:
        rows[(C.STRATEGY_LABELS[s], 'honest')] = compute_metrics(res_us[s].committed)
        rows[(C.STRATEGY_LABELS[s], 'lag 0')] = compute_metrics(simulate(s, px_us, sel_us, C.COST_BPS['us'], lag=0).committed)
        if s != 'distance':
            rows[(C.STRATEGY_LABELS[s], 'peeking')] = compute_metrics(simulate(s, px_us, peek, C.COST_BPS['us']).committed)
    bias = pd.DataFrame(rows).T[['Ann. return %', 'Sharpe', 'NW t-stat', 'Max DD %']]
    bias.to_csv('results/m1_lookahead_bias.csv'); display(bias.round(2))

Ann. return %  Sharpe  NW t-stat  \
S0 Distance (Gatev 2006)            honest          -0.090  -0.050     -0.210   
                                    lag 0            0.120   0.060      0.270   
S1 Cointegration z-score            honest           0.330   0.190      0.860   
                                    lag 0            0.430   0.240      1.100   
                                    peeking          2.690   2.840      9.420   
S2 OU s-score (Avellaneda-Lee 2010) honest           0.100   0.060      0.250   
                                    lag 0            0.160   0.090      0.390   
                                    peeking          5.060   3.600     13.330   
S3 OU optimal bands (Bertram 2010)  honest           0.120   0.060      0.270   
                                    lag 0            0.160   0.080      0.360   
                                    peeking          7.410   3.650     14.780   

                                             Max DD %  
S0 Distance (Gatev 2006)            honest     -5.780  
                                    lag 0      -5.140  
S1 Cointegration z-score            honest     -3.940  
                                    lag 0      -3.970  
                                    peeking    -1.390  
S2 OU s-score (Avellaneda-Lee 2010) honest     -4.160  
                                    lag 0      -4.410  
                                    peeking    -1.200  
S3 OU optimal bands (Bertram 2010)  honest     -4.840  
                                    lag 0      -4.270  
                                    peeking    -1.820

In [23]:
if RUN_HEAVY:
    # E5b (M4): trade-level evidence. One Sharpe ratio over 16 years has little power; ~1,800 round trips have more.
    # t-statistics are clustered by walk-forward window, because trades in the same window share market moves.
    import statsmodels.api as sm

    def clustered_mean(y, groups):
        fit = sm.OLS(np.asarray(y, float), np.ones(len(y))).fit(cov_type='cluster', cov_kwds={'groups': np.asarray(groups)})
        return float(fit.params[0]), float(fit.tvalues[0])

    last_day = {sel['window'].i: px_us.index[sel['window'].t1 - 1] for sel in sel_us}
    rows, exits = {}, {}
    for s in STRATS:
        for cost in [0, C.COST_BPS['us']]:
            t = (res_us[s] if cost else simulate(s, px_us, sel_us, 0)).trades.copy()
            t['window'] = t['pair'].str.slice(1, 3).astype(int)
            m, tm = clustered_mean(1e4 * t['pnl'], t['window'])
            h, th = clustered_mean((t['pnl'] > 0) - 0.5, t['window'])
            rows[(C.STRATEGY_LABELS[s], f'{cost} bp')] = {
                'Trades': len(t), 'Mean P&L per trade (bp)': m, 't (mean)': tm,
                'Hit rate %': 100 * (h + 0.5), 't (hit vs 50%)': th}
            if cost and s != 'distance':
                # Where the money goes: exits by the rule vs positions force-closed at the window's last day
                t['exit'] = np.where(t['close'] == t['window'].map(last_day), 'forced at window end', 'rule exit')
                for e, g in t.groupby('exit'):
                    exits[(C.STRATEGY_LABELS[s], e)] = {
                        'Share of trades %': 100 * len(g) / len(t), 'Hit rate %': 100 * (g['pnl'] > 0).mean(),
                        'Mean P&L per trade (bp)': 1e4 * g['pnl'].mean(), 'Total P&L (bp of pair capital)': 1e4 * g['pnl'].sum()}
    e5_trades = pd.DataFrame(rows).T
    e5_exits = pd.DataFrame(exits).T
    e5_trades.to_csv('results/e5_trades.csv'); e5_exits.to_csv('results/e5_exits.csv')
    hl = pd.cut(res_us['sscore'].pairs['half_life'], [0, 10, 21, 63]).value_counts().sort_index()
    print('Selected pair-windows by formation half-life (days):', dict(zip(hl.index.astype(str), hl.values)))
    display(e5_trades.round(2)); display(e5_exits.round(1))

Selected pair-windows by formation half-life (days): {'(0, 10]': np.int64(242), '(10, 21]': np.int64(17), '(21, 63]': np.int64(0)}


Trades  Mean P&L per trade (bp)  \
S0 Distance (Gatev 2006)            0 bp    651.000                   15.770   
                                    10 bp   651.000                   -4.230   
S1 Cointegration z-score            0 bp    424.000                   44.100   
                                    10 bp   424.000                   24.100   
S2 OU s-score (Avellaneda-Lee 2010) 0 bp    750.000                   24.170   
                                    10 bp   750.000                    4.170   
S3 OU optimal bands (Bertram 2010)  0 bp  1,577.000                    9.000   
                                    10 bp   618.000                    5.940   

                                           t (mean)  Hit rate %  \
S0 Distance (Gatev 2006)            0 bp      0.930      56.530   
                                    10 bp    -0.250      53.460   
S1 Cointegration z-score            0 bp      2.010      64.860   
                                    10 bp     1.100      62.740   
S2 OU s-score (Avellaneda-Lee 2010) 0 bp      1.860      71.470   
                                    10 bp     0.320      68.930   
S3 OU optimal bands (Bertram 2010)  0 bp      1.420      68.170   
                                    10 bp     0.370      63.920   

                                           t (hit vs 50%)  
S0 Distance (Gatev 2006)            0 bp            2.790  
                                    10 bp           1.470  
S1 Cointegration z-score            0 bp            7.460  
                                    10 bp           5.710  
S2 OU s-score (Avellaneda-Lee 2010) 0 bp           12.650  
                                    10 bp          10.240  
S3 OU optimal bands (Bertram 2010)  0 bp           14.640  
                                    10 bp           9.130

Share of trades %  \
S1 Cointegration z-score            forced at window end             45.300   
                                    rule exit                        54.700   
S2 OU s-score (Avellaneda-Lee 2010) forced at window end             28.400   
                                    rule exit                        71.600   
S3 OU optimal bands (Bertram 2010)  forced at window end             41.900   
                                    rule exit                        58.100   

                                                          Hit rate %  \
S1 Cointegration z-score            forced at window end      20.800   
                                    rule exit                 97.400   
S2 OU s-score (Avellaneda-Lee 2010) forced at window end      15.500   
                                    rule exit                 90.100   
S3 OU optimal bands (Bertram 2010)  forced at window end      17.400   
                                    rule exit                 97.500   

                                                          Mean P&L per trade (bp)  \
S1 Cointegration z-score            forced at window end                 -278.400   
                                    rule exit                             274.400   
S2 OU s-score (Avellaneda-Lee 2010) forced at window end                 -340.400   
                                    rule exit                             140.800   
S3 OU optimal bands (Bertram 2010)  forced at window end                 -329.000   
                                    rule exit                             247.600   

                                                          Total P&L (bp of pair capital)  
S1 Cointegration z-score            forced at window end                     -53,451.000  
                                    rule exit                                 63,669.300  
S2 OU s-score (Avellaneda-Lee 2010) forced at window end                     -72,500.600  
                                    rule exit                                 75,627.500  
S3 OU optimal bands (Bertram 2010)  forced at window end                     -85,218.700  
                                    rule exit                                 88,892.200

**E5 interpretation (M4).** US, 2011-2026, 10 bp per leg (results/table1_us.csv, e5_trades.csv, e5_exits.csv).

- **Result.** S1 does best (Sharpe 0.19) and the distance rule S0 loses slightly, but no rule's Sharpe ratio can be told apart from zero (standard error 0.25). The returns do not depend on the market (beta about 0.02).
- **The signal is real.** Before costs S1 earns about 44 bp per trade, and even after costs the rules win about two trades in three. The trading rules are not the problem.
- **Broken pairs lose the money.** Trades that the rule closes itself almost always win. But 28-45% of trades never come back before the six-month window ends, and these lose about 3% each, which cancels the gains. Only about 8 of 196 pairs pass Engle-Granger per window, roughly what chance alone would give, so many selected "pairs" are not real pairs (E9 checks this directly).

## 9. E8 India  (Owners: M2 India selection, M5 India table)  HEAVY
The same pipeline on 26 NSE stocks in 5 sectors, 2012-2026, at 15 bp per leg.

In [24]:
if RUN_HEAVY:
    sel_in = run_selection('india', px_in, sectors('india'))
    res_in = {s: simulate(s, px_in, sel_in, C.COST_BPS['india'], label=s) for s in STRATS}
    table_in = metrics_table(res_in.values(), bench_in).rename(index=C.STRATEGY_LABELS)
    table_in.to_csv('results/table_india.csv'); display(table_in.round(2))

,Ann. return %,Ann. vol %,Sharpe,Sharpe SE (Lo),Sharpe SE (HAC),NW t-stat,Sortino,Max DD %,Calmar,Total return %,Years,Market beta,Market corr,Trades,Hit rate %,Avg days held
S0 Distance (Gatev 2006),0.430,2.700,0.160,0.270,0.260,0.610,0.230,-4.600,0.090,5.410,13.430,0.010,0.050,472.000,53.810,56.880
S1 Cointegration z-score,0.410,1.070,0.380,0.270,0.250,1.510,0.600,-3.250,0.120,5.530,13.430,0.000,0.000,121.000,67.770,47.010
S2 OU s-score (Avellaneda-Lee 2010),0.370,1.150,0.320,0.270,0.240,1.330,0.500,-2.510,0.150,5.000,13.430,-0.000,-0.000,234.000,68.380,29.620
S3 OU optimal bands (Bertram 2010),0.430,1.340,0.320,0.270,0.240,1.310,0.480,-3.600,0.120,5.790,13.430,-0.000,-0.000,203.000,69.460,49.660


In [25]:
if RUN_HEAVY:
    # T31 (M2): the India universe through pair selection (sel_in is cached in results/cache/ after the first run)
    from src.backtest import CACHE
    from src.selection import candidate_pairs
    assert (CACHE / f'selection_v2_india_{C.FORMATION_DAYS}_{C.TRADING_DAYS}_{C.TOP_K}.pkl').exists()
    n_cand = len(candidate_pairs(sectors('india'), set(px_in.columns)))
    diag_sel = pd.DataFrame([s['diag'] for s in sel_in])
    chosen_in = [(s['window'], p) for s in sel_in for p in s['coint']]
    admitted = [int((px_in.iloc[s['window'].f0:s['window'].t0].isna().mean() <= 0.02).sum()) for s in sel_in]
    # every selected pair satisfies every filter, within one sector, and trades only after its formation window
    assert all(p.eg_pvalue < C.EG_PVALUE and p.beta > 0 and C.HALF_LIFE_RANGE[0] <= p.half_life <= C.HALF_LIFE_RANGE[1]
               and p.crossings >= C.MIN_CROSSINGS_PER_YEAR for _, p in chosen_in)
    assert all(s['window'].t0 - s['window'].f0 == C.FORMATION_DAYS and len(s['coint']) <= C.TOP_K for s in sel_in)
    india_sel = pd.Series({
        'Walk-forward windows': len(sel_in),
        'First / last trading day': f"{px_in.index[sel_in[0]['window'].t0].date()} / {px_in.index[sel_in[-1]['window'].t1 - 1].date()}",
        'Candidate within-sector pairs per window': n_cand,
        'Tickers admitted per window (min)': min(admitted),
        'Pairs passing Engle-Granger per window': diag_sel.n_eg.mean(),
        'Expected by chance alone (5% of candidates)': 0.05 * n_cand,
        'Pairs surviving Benjamini-Hochberg per window': diag_sel.n_bh.mean(),
        'Pairs passing all filters per window': diag_sel.n_filtered.mean(),
        'Windows with no pair': int((diag_sel.n_filtered == 0).sum()),
        'Pairs selected in total': len(chosen_in),
        'Longest fitted half-life (days)': max(p.half_life for _, p in chosen_in)}, name='India selection')
    india_sel.to_csv('results/india_selection_summary.csv')
    display(india_sel)
    display(pd.Series([p.sector for _, p in chosen_in]).value_counts().rename('Selected pairs by sector'))

Walk-forward windows                                                  27
First / last trading day                         2013-01-10 / 2026-09-29
Candidate within-sector pairs per window                              55
Tickers admitted per window (min)                                     26
Pairs passing Engle-Granger per window                             3.222
Expected by chance alone (5% of candidates)                        2.750
Pairs surviving Benjamini-Hochberg per window                      1.000
Pairs passing all filters per window                               3.074
Windows with no pair                                                   4
Pairs selected in total                                               83
Longest fitted half-life (days)                                   11.560
Name: India selection, dtype: object

Banks     42
Auto      12
FMCG      11
IT        10
Pharma     8
Name: Selected pairs by sector, dtype: int64

**T31 India selection (M2).** (results/india_selection_summary.csv) `run_selection('india')` runs through 27 walk-forward windows (first trading day 10 Jan 2013, last 29 Sep 2026) and is cached. All 26 stocks have enough formation-window data in every window, and each window tests the 55 within-sector pairs (15 in Banks, 10 in each of the other four sectors). On average 3.2 pairs pass Engle-Granger against 2.75 expected by chance, Benjamini-Hochberg keeps 1.0, and 3.1 pairs pass every filter, so only about 3 of the 20 slots are used and 4 windows trade nothing. Of the 83 pairs selected in total, 42 are banks. As in the US, the screen finds little more than a 5% test would find by chance, which is the context for M5's India results.

In [26]:
if RUN_HEAVY:
    # E8 India (M5): how many pairs pass, the test-block split, and trade-level evidence, as for the US (E5, E5b)
    import statsmodels.api as sm
    diag_in = pd.DataFrame([s['diag'] for s in sel_in])
    print(f"India, per window: {diag_in.n_tested.mean():.0f} pairs tested, {diag_in.n_eg.mean():.1f} pass Engle-Granger "
          f"(chance alone at 5%: up to {0.05 * diag_in.n_tested.mean():.1f}), {diag_in.n_bh.mean():.1f} survive "
          f"Benjamini-Hochberg, {diag_in.n_filtered.mean():.1f} traded; {int((diag_in.n_filtered == 0).sum())} windows with no pair")
    rows = {}
    for s, r in res_in.items():
        t = r.trades.copy()
        t['window'] = t['pair'].str.slice(1, 3).astype(int)
        fit = sm.OLS((t['pnl'] > 0) - 0.5, np.ones(len(t))).fit(cov_type='cluster', cov_kwds={'groups': t['window']})
        row = {'Trades': len(t), 'Hit rate %': 100 * (fit.params.iloc[0] + 0.5), 't (hit vs 50%)': fit.tvalues.iloc[0],
               'Mean P&L per trade (bp)': 1e4 * t['pnl'].mean()}
        for blk, sl in [('2013-2026', slice(None)), ('2013-2021', slice(None, '2021-12-31')),
                        ('2022-2026 test', slice(C.TEST_START, None))]:
            row[f'Sharpe {blk}'] = compute_metrics(r.committed[sl])['Sharpe']
        rows[C.STRATEGY_LABELS[s]] = row
    e8_india = pd.DataFrame(rows).T
    e8_india.to_csv('results/e8_india_detail.csv'); display(e8_india.round(2))

India, per window: 55 pairs tested, 3.2 pass Engle-Granger (chance alone at 5%: up to 2.8), 1.0 survive Benjamini-Hochberg, 3.1 traded; 4 windows with no pair


,Trades,Hit rate %,t (hit vs 50%),Mean P&L per trade (bp),Sharpe 2013-2026,Sharpe 2013-2021,Sharpe 2022-2026 test
S0 Distance (Gatev 2006),472.000,53.810,1.640,24.410,0.160,0.260,-0.090
S1 Cointegration z-score,121.000,67.770,3.620,90.310,0.380,0.430,0.280
S2 OU s-score (Avellaneda-Lee 2010),234.000,68.380,5.760,42.490,0.320,0.290,0.560
S3 OU optimal bands (Bertram 2010),203.000,69.460,6.130,56.670,0.320,0.220,0.820


**E8 India result (M5).** India behaves like the US. All four rules are slightly positive (Sharpe 0.16-0.38, S1 best) but none significantly, and only about 3 of 55 pairs pass Engle-Granger per window, no more than chance would give. Per trade, the rules again win about two trades in three (results/table_india.csv, results/e8_india_detail.csv).

## 10. E9 do ADF-selected pairs stay cointegrated?  (Owner: M2)  HEAVY
For every selected pair we re-test the trading-window spread, built with the formation $\beta$, using ADF (no $\beta$ is re-estimated, so plain ADF critical values are correct here).

In [27]:
if RUN_HEAVY:
    # E9 (M2, T32): do the pairs selected on the formation window stay cointegrated in the trading window?
    from scipy.stats import spearmanr
    logp_us = np.log(px_us)
    rows = []
    for sel in sel_us:
        w = sel['window']
        for s in sel['coint']:
            x = (logp_us[s.a] - s.beta * logp_us[s.b] - s.mu).iloc[w.t0:w.t1].dropna().values
            if len(x) > 40:
                rows.append(dict(window=w.i, pair=f'w{w.i:02d}:{s.a}/{s.b}', eg_stat=s.eg_stat,
                                 half_life=s.half_life, still_coint=adf_test(x, maxlag=5)[1] < 0.05))
    e9 = pd.DataFrame(rows).merge(res_us['sscore'].pairs[['pair', 'pnl']], on='pair')
    # What share would pass if EVERY pair were still an OU process with its fitted half-life? ADF power at 126 days (E2a)
    hl_grid = np.log(np.asarray(power.index, dtype=float))
    e9['expected_pass'] = np.interp(np.log(e9.half_life), hl_grid, power[126].values)
    e9['decile'] = pd.qcut(e9.eg_stat, 10, labels=False) + 1
    obs, exp_ = e9.still_coint.mean(), e9.expected_pass.mean()
    print(f'{len(e9)} selected pair-windows: {100 * obs:.1f}% pass ADF in the trading window; '
          f'{100 * exp_:.1f}% would pass if all were still OU (power at 126 days, E2a); 5% if none were')
    rho, p_rho = spearmanr(e9.eg_stat, e9.pnl)
    print(f'Spearman correlation of formation EG statistic with pair P&L: {rho:+.3f} (p = {p_rho:.2f})')

    by_dec = e9.groupby('decile').agg(n=('pnl', 'size'), mean_eg_stat=('eg_stat', 'mean'),
                                      pnl_pct=('pnl', lambda v: 100 * v.mean()),
                                      pnl_se_pct=('pnl', lambda v: 100 * v.std() / np.sqrt(len(v))),
                                      still_coint=('still_coint', 'mean'), expected=('expected_pass', 'mean'))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    axes[0].bar(by_dec.index - 0.2, 100 * by_dec.still_coint, width=0.4, color=COLORS['sscore'], label='Observed')
    axes[0].bar(by_dec.index + 0.2, 100 * by_dec.expected, width=0.4, color='grey', alpha=0.6, label='If all still OU')
    axes[0].axhline(5, color='k', lw=0.8, ls=':')
    axes[0].set_xlabel('Formation EG statistic decile (1 = most negative)'); axes[0].set_ylabel('Pairs passing ADF in trading window (%)')
    axes[0].legend(fontsize=8)
    axes[1].bar(by_dec.index, by_dec.pnl_pct, yerr=2 * by_dec.pnl_se_pct, capsize=2, color=COLORS['sscore'])
    axes[1].axhline(0, color='k', lw=0.8)
    axes[1].set_xlabel('Formation EG statistic decile (1 = most negative)'); axes[1].set_ylabel('Mean S2 pair P&L per window (%)')
    savefig(fig, 'fig_e9_persistence')
    e9.to_csv('results/e9_persistence.csv', index=False)
    by_dec.to_csv('results/e9_by_decile.csv')
    display(by_dec.round(3))
    display(e9.groupby('still_coint').pnl.describe().round(4))

259 selected pair-windows: 9.3% pass ADF in the trading window; 51.8% would pass if all were still OU (power at 126 days, E2a); 5% if none were
Spearman correlation of formation EG statistic with pair P&L: -0.015 (p = 0.81)


/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, *_ = adfuller(np.asarray(x, dtype=float), regression=regression,
/Users/satya_lakshya/projects/Stochastic Calculus/Stochastic Calculus 3-1 project/Pairs-Trading/src/cointegration.py:33: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the curre

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


,n,mean_eg_stat,pnl_pct,pnl_se_pct,still_coint,expected
decile,,,,,,
1,26,-5.113,1.322,1.212,0.077,0.659
2,26,-4.427,-0.952,0.949,0.077,0.602
3,26,-4.238,-0.218,1.145,0.077,0.585
4,26,-4.083,1.501,1.293,0.115,0.525
5,26,-3.964,-0.142,1.267,0.077,0.508
6,25,-3.849,-1.213,1.335,0.080,0.499
7,26,-3.749,0.472,1.294,0.115,0.497
8,26,-3.662,-0.227,1.031,0.115,0.454
9,26,-3.549,0.332,1.036,0.115,0.420


,count,mean,std,min,25%,50%,75%,max
still_coint,,,,,,,,
False,235.000,0.001,0.058,-0.244,-0.026,0.002,0.029,0.234
True,24.000,-0.001,0.056,-0.128,-0.033,0.006,0.031,0.150


**E9 result (M2).** (results/e9_persistence.csv, results/e9_by_decile.csv, figs/fig_e9_persistence)

- **Most selected pairs do not stay cointegrated.** Of 259 selected pair-windows, only 24 (9.3%) pass ADF on the trading-window spread (formation $\beta$, so plain ADF critical values are correct). A 126-day window has limited power, so we compare with what we would see if every pair were still an OU process: using E2a's power at each pair's fitted half-life, about 52% would pass. Observed persistence is far below that and close to the 5% of a test with nothing to find.
- **A stronger formation statistic does not help.** Pairs in the most negative EG-statistic decile persist no more often than the weakest (8% against 8%), and P&L is flat across deciles (Spearman $\rho = -0.02$, $p = 0.81$; two-standard-error bars cover zero in every decile). Pairs that stay cointegrated earn no more than those that do not (mean 0.14% for broken pairs, -0.05% for persistent ones, with a standard deviation of about 6% per pair-window).
- **Link to E5.** This is the direct evidence behind M4's finding in E5 that losses come from pairs whose relationship broke: the formation test selects mostly pairs that look cointegrated by chance, consistent with E2b (a true 5% test still flags about 10 of 196 independent pairs) and with Engle-Granger passing only about as many pairs as chance in E5.
- **Caveats.** The ADF test on 126 days cannot confirm slow reversion (E2a), and fitted half-lives are biased short (E1), so the expected share above is, if anything, too high. A pair that fails the test in the trading window can still have traded profitably in it.

## 11. Conclusions  (Owner: M3)
1. **Detection.** With one year of data ADF reliably finds only fast mean reversion (half-lives up to about 10 days). Engle-Granger passes about as many pairs as chance would, and most of them stop mean-reverting soon after selection (E2a, E9).
2. **Model.** The OU model and Bertram's formula are correct, but estimating $\kappa$ is hard: with one year of data $\hat\kappa$ is about 18% too high even for a fast pair, and a perfect synthetic pair earns only about half its theoretical return once realistic trading is applied (E1, E3, E4).
3. **Trading.** Before costs the rules catch real mean reversion (they win about two trades in three), but after 10 bp per leg no rule earns a significant return in the US or India; the best, S1, earns about twice the round-trip cost per trade before costs (E5, E8). The weak link is pair selection, not the trading rules.

*Limitations: survivorship bias (today's large caps), a fixed hedge ratio within each window, daily closes only. Future work: a larger set of stocks, a stricter or more stable pair selection, and a time-varying hedge ratio.*